# Stage 3 — Catheter / guidewire tracking (Colab/Kaggle BUILD)

**Thin notebook** — logic in `src/*`. YOLO11n (2 classes: catheter, guidewire) on **CathAction**, then **ByteTrack** for continuity. ByteTrack is Python around the detector, not inside the CoreML graph.

Acceptance = **fps + ID-switches**, not just IoU.

**Dataset:** CathAction `airvlab/CathAction` (HF). This notebook auto-pulls **`segmentation_human_train.zip`** (143 MB, human catheter+guidewire masks) — the right split for a clinical detector. The 41.8 GB action + 4.35 GB collision zips are other tasks; skip them.

In [1]:
# 0) Repo + env. Colab mounts Drive so repo+runs persist; Kaggle uses /kaggle/working.
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/intv-img/interventional-imaging-pipeline'
except Exception:
    REPO = '/kaggle/working/interventional-imaging-pipeline'
REPO_URL = 'https://github.com/jugalmodi0111/interventional-imaging-pipeline.git'
if not os.path.exists(REPO):
    !git clone {REPO_URL} {REPO}
else:
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main  # pull latest src so the honest split fix is present (avoid stale Drive repo)
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install "ultralytics>=8.2" pycocotools opencv-python pyyaml huggingface_hub
from src import env
E = env.setup()
assert E['device'] == 'cuda', 'Switch runtime to GPU.'
import torch
print('catheter build env ready | device', E['device'], '|', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

/kaggle/working/interventional-imaging-pipeline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 48.6 kB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.3 MB/s eta 0:00:00:00:0100:01
[env] kaggle | device=cuda | root=/kaggle/working/intv-img
catheter build env ready | device cuda | Tesla T4
Tesla T4, 15360 MiB, 3 MiB
Tesla T4, 15360 MiB, 3 MiB


In [2]:
# 1) Get CathAction into data/raw/cathaction. Layout = a root holding img/ + mask/ (human_dataset_train).
#    Priority: a Kaggle-attached copy -> else download the 143 MB human seg split from HF.
import os, glob, zipfile

def _img_mask_root(base):
    for md in glob.glob(os.path.join(base, "**", "mask"), recursive=True):
        r = os.path.dirname(md)
        if os.path.isdir(os.path.join(r, "img")):
            return r
    return None

os.makedirs("data/raw", exist_ok=True)
src = _img_mask_root("/kaggle/input") if os.path.exists("/kaggle/input") else None
if src and not os.path.exists("data/raw/cathaction"):
    os.symlink(src, "data/raw/cathaction"); print("using Kaggle input root:", src)
os.makedirs("data/raw/cathaction", exist_ok=True)

if not _img_mask_root("data/raw/cathaction"):
    from huggingface_hub import hf_hub_download
    zp = hf_hub_download(repo_id="airvlab/CathAction", repo_type="dataset",
                         filename="segmentation_human_train.zip")   # 143 MB, human catheter/guidewire masks
    with zipfile.ZipFile(zp) as z: z.extractall("data/raw/cathaction")
    print("downloaded + unzipped ->", zp)
# more data (optional): also fetch filename="segmentation_animal_phantom.zip" (9.88 GB) the same way.

root = _img_mask_root("data/raw/cathaction")
imgs = glob.glob(os.path.join(root, "img", "*")) if root else []
print("CathAction root:", root, "| img frames:", len(imgs))
assert root and imgs, "CathAction img/+mask/ not found - check Kaggle attach / HF download layout"

using Kaggle input root: /kaggle/input/datasets/jugalmodi1103/segmentation-human-train-new/human_dataset_train
CathAction root: data/raw/cathaction | img frames: 5283


In [3]:
# 2) CathAction -> YOLO (catheter, guidewire). Importable converter (COCO or mask fallback).
import yaml
from src.data_prep import cathaction_to_yolo
cfg = yaml.safe_load(open('configs/catheter_track.yaml'))
cathaction_to_yolo.main(cfg)

CathAction -> data/processed/catheter : 5225 frames ; data cfg data/processed/catheter/data.yaml


In [4]:
# 3) Train the 2-class detector on GPU (reuses the generalized train_detector).
from src.train.train_detector import train
best = train(cfg, project=f"{E['runs']}/catheter", device=0)  # device=0 = force GPU, fail loud if none
print('best ->', best)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/catheter/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s

In [5]:
# 3b) (Optional) Build a real sample_clip.mp4 from one frame sequence, so tracking runs on an
#     actual video file. Skip if you attached video_action_understanding.zip (has procedure mp4s).
import glob, os, cv2
from collections import Counter
OUT_MP4 = 'data/raw/cathaction/sample_clip.mp4'
IMG = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
BAD = ('mask', 'guidewire', 'label', 'annotation', 'gt', '/seg')
have_mp4 = os.path.exists(OUT_MP4) or bool(glob.glob('data/raw/cathaction/**/*.mp4', recursive=True))
if not have_mp4:
    fr = [p for p in glob.glob('data/raw/cathaction/**/*', recursive=True) if p.lower().endswith(IMG)]
    fr = [p for p in fr if not any(b in p.lower().split('cathaction/', 1)[-1] for b in BAD)]
    if fr:
        d   = Counter(os.path.dirname(p) for p in fr).most_common(1)[0][0]   # biggest dir = one run
        seq = sorted(p for p in fr if os.path.dirname(p) == d)
        h, w = cv2.imread(seq[0]).shape[:2]
        vw = cv2.VideoWriter(OUT_MP4, cv2.VideoWriter_fourcc(*'mp4v'), 15, (w, h))
        for p in seq:
            im = cv2.imread(p)
            vw.write(im if im.shape[:2] == (h, w) else cv2.resize(im, (w, h)))
        vw.release()
        print(f'built {OUT_MP4} from {len(seq)} frames @ {d}')
    else:
        print('no frames found to build mp4 (cell 4 will fall back to frame-dir tracking)')
else:
    print('mp4 already present -> skip build')

built data/raw/cathaction/sample_clip.mp4 from 5283 frames @ data/raw/cathaction/img


In [6]:
# 4) Track with ByteTrack. The seg split has NO video, so run on a frame SEQUENCE folder:
#    Ultralytics accepts an image dir and processes frames in sorted (=temporal) order for ID continuity.
import glob, os
from collections import Counter
from src.serve.track import track_yolo
IMG = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
BAD = ('mask', 'guidewire', 'label', 'annotation', 'gt', '/seg')   # skip label/mask dirs (not raw frames)

clip = 'data/raw/cathaction/sample_clip.mp4'
if not os.path.exists(clip):
    mp4s = glob.glob('data/raw/cathaction/**/*.mp4', recursive=True)
    clip = mp4s[0] if mp4s else None
if clip is None:
    frames = [p for p in glob.glob('data/raw/cathaction/**/*', recursive=True)
              if p.lower().endswith(IMG)]
    frames = [p for p in frames
              if not any(b in p.lower().split('cathaction/', 1)[-1] for b in BAD)]
    if frames:                                    # pick the most-populated dir = one imaging run
        clip = Counter(os.path.dirname(p) for p in frames).most_common(1)[0][0]
        k = sum(1 for p in frames if os.path.dirname(p) == clip)
        print(f'no mp4 -> tracking on frame dir: {clip} ({k} frames). Verify it is one raw sequence.')
if clip:
    track_yolo(best, source=clip, out=f"{E['runs']}/catheter/track.mp4", device=0)
else:
    print('no frames/mp4 under data/raw/cathaction — skipping ByteTrack demo (best.pt still exported)')

no mp4 -> tracking on frame dir: data/raw/cathaction/img (5283 frames). Verify it is one raw sequence.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 350ms
Prepared 1 package in 103ms
Installed 1 package in 6ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

frames 5283 | unique track ids 304 | audit -> runs/audit.jsonl


## Handoff
Copy `best.pt` to the Mac: `make export-coreml-yolo MODEL=best.pt` → `.mlpackage`, then per-frame edge detection via `src.serve.realtime --task det`; ByteTrack stays in the app loop (`src.serve.track`).

**Persist (Kaggle):** `/kaggle/working` is wiped on session death — use *Save & Run All (Commit)* or download `best.pt` before closing. Colab repo lives on Drive → runs persist.